In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# 1. Load data
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

# 2. Advanced Feature Engineering (from Sniper Elite)
def advanced_features(df):
    df_eng = df.copy()
    # Interaction Âge x Pages
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    # Utilisateur Actif
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    # Vitesse de consommation
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

target = 'converted'
X = advanced_features(train_data.drop(target, axis=1))
y = train_data[target]
X_test_final = advanced_features(test_data)

# 3. Categorical Encoding
categorical_cols = ['country', 'source']
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test_final[col]]))
    X[col] = le.transform(X[col])
    X_test_final[col] = le.transform(X_test_final[col])

# 4. Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Train Poisson Monster Evolution
model = HistGradientBoostingRegressor(
    loss='poisson',
    learning_rate=0.05,
    max_iter=500,
    max_depth=8,
    l2_regularization=0.1,
    categorical_features=cat_indices,
    random_state=42
)

model.fit(X_train, y_train)

# 6. Threshold Tuning
y_scores = model.predict(X_val)
best_f1 = 0
best_threshold = 0
thresholds = np.linspace(y_scores.min(), y_scores.max(), 300)

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

# 7. Metrics
y_preds = (y_scores >= best_threshold).astype(int)
auc = roc_auc_score(y_val, y_scores)
prec = precision_score(y_val, y_preds)
rec = recall_score(y_val, y_preds)

print(f"--- Poisson Monster Evolution Results ---")
print(f"F1-Score: {best_f1:.6f}")
print(f"ROC-AUC: {auc:.6f}")
print(f"Precision: {prec:.6f}")
print(f"Recall: {rec:.6f}")
print(f"Best Threshold: {best_threshold:.6f}")

# 8. Submission
test_scores = model.predict(X_test_final)
test_preds = (test_scores >= best_threshold).astype(int)
submission = pd.DataFrame({'converted': test_preds})
submission.to_csv('submission_POISSON_EVOLUTION.csv', index=False)
print(f"\nSaved 'submission_POISSON_EVOLUTION.csv' with {test_preds.sum()} conversions.")

--- Poisson Monster Evolution Results ---
F1-Score: 0.774081
ROC-AUC: 0.985575
Precision: 0.833962
Recall: 0.722222
Best Threshold: 0.428570

Saved 'submission_POISSON_EVOLUTION.csv' with 888 conversions.
